# Curry 1930《组合逻辑基础》第一部分：可检查的微型复现

原文：H. B. Curry, *Grundlagen der kombinatorischen Logik. Teil I*, American Journal of Mathematics 52(3), 509–536。这里仅复现第一部分中可直接检查的组合子机制与几条等式证明。

**请始终区分两层：**
1. 四条 B/C/W/K 规则的定向、带上下文归约。I 只是 W K 的缩写。
2. 单独的等式证书核：接纳原文的有限组合子公理，并使用 I.D 已建立的反身性、对称性、传递性和应用同余。

这不是完整 Q/Π/P/Λ 断言演算，不验证原体系的一致性、完备性，也不复现第二部分。有限测试、额外参数后的相同行为，都不能替代未加参数的等式证明。

**执行记录说明：** 本文件的输出由随包 execute_notebook.py 使用 Python 标准库 exec 逐格执行并捕获 stdout/stderr 生成，使用同一命名空间；没有启动 Jupyter kernel。所有代码格都是普通 Python，显式 print 输出。可在 Jupyter 中自行 Run All，也可无额外依赖用该脚本复跑。

In [1]:
from pathlib import Path
import platform
if not Path("curry.py").is_file():
    raise RuntimeError("请在本 notebook 所在目录运行，确保 curry.py 同目录。")
from curry import (Atom, B, C, W, K, I, app, parse, pretty, normal, normalize,
                   sequence, dot, AXIOMS, AXIOM_ARITIES, BW_P521_LITERAL,
                   check, axiom, Proof, InvalidProof, by_reduction,
                   cbi_identity_proof, right_identity_proof,
                   associativity_proof, axiom_names_used, proof_nodes)
from run_demo import show_trace, show_axiom_actions, show_certificate
print("Python:", platform.python_version())
print("Runtime dependencies: standard library only")
print("Saved execution method: stdlib exec; Jupyter kernel used: False")


Python: 3.12.14
Runtime dependencies: standard library only
Saved execution method: stdlib exec; Jupyter kernel used: False


## 1. 应用是左结合，I 是定义

p.520 的应用记法与 p.521 的 I≡WK 对应不可变 Atom/App 树。`x y z` 是 `((x y) z)`，不同于 `x (y z)`。大写字母可紧邻，`BBC` 会读作 `B B C`；小写标识符以空格分开。`B2` 只是一个原子；请用 sequence("B", 2) 获取 B₂。

In [2]:
assert parse("BXY") == app(B, Atom("X"), Atom("Y"))
assert parse("x y z") != parse("x (y z)")
assert parse("I") == app(W, K)
print("x y z tree:", repr(parse("x y z")))
print("x (y z) tree:", repr(parse("x (y z)")))
print("I expands to:", pretty(I))


x y z tree: App(fn=App(fn=Atom(name='x'), arg=Atom(name='y')), arg=Atom(name='z'))
x (y z) tree: App(fn=Atom(name='x'), arg=App(fn=Atom(name='y'), arg=Atom(name='z')))
I expands to: W K


## 2. 四条规则与可核查轨迹

p.522：Bxyz→x(yz)，Cxyz→xzy，Wxy→xyy，Kxy→x。原文把它们作为等式生成规则；箭头方向是本复现的计算选择。

策略为完整上下文中的最左最外层归约，会进入参数内部；这不是仅到弱头正规形就停的求值器。路径中的 0 表示函数子树，1 表示参数子树。轨迹的每一步都由不调用 reducer 匹配器的证书核再次检查。

In [3]:
for example in ("B f g x", "C f x y", "W f x", "K x y", "I x", "f (I x)"):
    show_trace(example)
    print()


start: B f g x  [expanded: B f g x]
  01. B@root: f (g x)
status=normal; contractions=1

start: C f x y  [expanded: C f x y]
  01. C@root: f y x
status=normal; contractions=1

start: W f x  [expanded: W f x]
  01. W@root: f x x
status=normal; contractions=1

start: K x y  [expanded: K x y]
  01. K@root: x
status=normal; contractions=1

start: I x  [expanded: W K x]
  01. W@root: K x x
  02. K@root: x
status=normal; contractions=2

start: f (I x)  [expanded: f (W K x)]
  01. W@1: f (K x x)
  02. K@1: f x
status=normal; contractions=2



## 3. Bₙ、Cₙ、Wₙ、Kₙ

原文 pp.529、531–532：B₀≡I，B₁≡B，Bₙ₊₁≡B B Bₙ（n≥1）。C₁≡C、Cₙ₊₁≡B Cₙ；W、K 同理。

要注意：B₁=B B B₀ 是 p.531 的等式定理，不是程序偷偷采用的语法定义。下面检验 n=0…6 的 B 作用式和 n=1…6 的 C/W/K 作用式；有限取值不等于计算证明对所有 n 的归纳定理。

In [4]:
checks = 0
f, g = Atom("f"), Atom("g")
for n in range(7):
    xs = tuple(Atom(f"x{i}") for i in range(n))
    assert normal(app(sequence("B", n), f, g, *xs)) == app(f, app(g, *xs))
    checks += 1
for n in range(1, 7):
    xs = tuple(Atom(f"x{i}") for i in range(n + 2))
    expected = (
        ("C", xs, app(xs[0], *xs[1:n], xs[n+1], xs[n])),
        ("W", xs[:-1], app(xs[0], *xs[1:n], xs[n], xs[n])),
        ("K", xs[:-1], app(xs[0], *xs[1:n])),
    )
    for kind, args, target in expected:
        assert normal(app(sequence(kind, n), *args)) == target
        checks += 1
        if n == 3:
            print(f"{kind}_3:", pretty(target))
print("Finite indexed action checks:", checks)


C_3: x0 x1 x2 x4 x3
W_3: x0 x1 x2 x3 x3
K_3: x0 x1 x2
Finite indexed action checks: 25


## 4. 十六条有限公理：检查已加参数的作用

pp.521、534–535 给出有限的闭项公理。本实验在每个公理两边附加足量的新鲜符号参数，直到两边归约为同一棵只含变量的树，并独立核验归约证书。

这些是**加参数后的等式实例**，不是通过测试“证明原公理”。证书核另将这些有限公理作为前提接纳。测试文件还用独立的 Python 高阶函数解释器检查这些输出，并逐侧对照 pp.534–535 的乘积记法。

In [5]:
show_axiom_actions()


16 finite axioms: symbolic saturated actions (not derivations of bare axioms)
  Ax.B   args=4; steps L/R=7/4; x0 (x1 (x2 x3))
  Ax.C   args=4; steps L/R=7/6; x0 (x1 x3 x2)
  Ax.W   args=3; steps L/R=5/6; x0 (x1 x2 x2)
  Ax.K   args=3; steps L/R=5/5; x0 x1
  Ax.I1  args=2; steps L/R=4/6; x0 x1
  Ax.BC  args=4; steps L/R=3/7; x0 x3 (x1 x2)
  Ax.BW  args=3; steps L/R=3/13; x0 (x1 x2) (x1 x2)
  Ax.BK  args=3; steps L/R=3/3; x0
  Ax.CC1 args=3; steps L/R=3/4; x0 x1 x2
  Ax.CC2 args=4; steps L/R=7/6; x0 x3 x2 x1
  Ax.CW  args=3; steps L/R=3/7; x0 x2 x2 x1
  Ax.CK  args=3; steps L/R=3/2; x0 x1
  Ax.WC  args=2; steps L/R=3/1; x0 x1 x1
  Ax.WW  args=2; steps L/R=3/4; x0 x1 x1 x1
  Ax.WK  args=2; steps L/R=3/3; x0 x1
  Ax.I2  args=2; steps L/R=3/2; x0 x1


## 5. 一个实际发现：Ax.(BW) 的两处写法

p.521 扫描的字面读法似为 `BBW = B(B(B(B(BW)W)(BC))B(BB))B`。按全篇左结合约定解释，下面两边作用不同。

本复现采用 p.534 明确的左结合乘积式：B₁·W₁ = W₂·W₁·C₂·(B₂ B)·B。展开并归约其中 B₂B 后，右边是 `B(B(B(B(BW)W)(BC))(B(BB)))B`。

两处差异已保留为回归测试。这里只报告扫描读法与后文写法的可重现差异，不声称发现了官方勘误，也不把本包的 BW 写法说成 p.521 的逐字录入。

In [6]:
xs = tuple(Atom(f"x{i}") for i in range(3))
literal_left, literal_right = BW_P521_LITERAL
left_nf = normal(app(literal_left, *xs))
right_nf = normal(app(literal_right, *xs))
print("p.521 literal left :", pretty(left_nf))
print("p.521 literal right:", pretty(right_nf))
assert left_nf != right_nf
print("p.534 selected RHS :", pretty(normal(app(AXIOMS["BW"][1], *xs))))
assert normal(app(AXIOMS["BW"][1], *xs)) == left_nf


p.521 literal left : x0 (x1 x2) (x1 x2)
p.521 literal right: x0 (x1 (B x0)) (x1 x2)
p.534 selected RHS : x0 (x1 x2) (x1 x2)


## 6. 为什么 BI=I 不能靠本 reducer 冒充证明？

I 展开为 WK。BI 与 I 都缺少足够的参数，四条规则下各自已是正规形，而且树不同。追加两个新鲜参数后行为相同，仍不能无条件从“作用相同”推出闭项相等。

原文明确给出 Ax.I₂：BI=I（pp.521、535）；p.536 讨论了这条公理与追加变量数量的关系。本复现只在等式证书层引用它，绝不把它暗藏为第五条归约规则。

In [7]:
x, y, z, u = map(Atom, "xyzu")
print("normal(B I):", pretty(normal(app(B, I))))
print("normal(I)  :", pretty(normal(I)))
assert normal(app(B, I)) != normal(I)
print("B I x y and I x y:", pretty(normal(app(B, I, x, y))), pretty(normal(app(I, x, y))))
show_certificate("Ax.I2", axiom("I2"))
show_certificate("p.531 CBI=I", cbi_identity_proof())
show_certificate("p.531 BxI=x", right_identity_proof(x))


normal(B I): B (W K)
normal(I)  : W K
B I x y and I x y: x y x y
Ax.I2: B (W K) = W K
  checked nodes=1; finite axioms=I2
p.531 CBI=I: C B (W K) = W K
  checked nodes=20; finite axioms=I1,I2
p.531 BxI=x: B x (W K) = x
  checked nodes=29; finite axioms=I1,I2


## 7. 乘积结合律：追加参数的实验，以及真正的等式证书

p.532 定义 X·Y≡BXY。p.533 的裸项结合律不是仅由四条定向规则把两边归约到相同正规形得到。

这里先展示追加 u 后的相同行为，再依照原文路线，用 Ax.B 的实例先证明 B(X·Y)=BX·BY，再证明 (X·Y)·Z=X·(Y·Z)。证书核显式使用对称、传递与应用同余，没有 η／外延性推断规则。

In [8]:
left, right = dot(dot(x, y), z), dot(x, dot(y, z))
print("bare left normal form :", pretty(normal(left)))
print("bare right normal form:", pretty(normal(right)))
assert normal(left) != normal(right)
print("after u:", pretty(normal(app(left, u))), "=", pretty(normal(app(right, u))))
proof = associativity_proof(x, y, z)
assert check(proof) == (left, right)
show_certificate("p.533 associativity", proof)
assert axiom_names_used(proof) == frozenset(("B",))


bare left normal form : B (B x y) z
bare right normal form: B x (B y z)
after u: x (y (z u)) = x (y (z u))
p.533 associativity: B (B x y) z = B x (B y z)
  checked nodes=47; finite axioms=B


## 8. 非终止、燃料与任意项实例化

WWW 的唯一可收缩项就是整体；W 规则把它改写回 WWW，所以这个具体例子确实没有四规则正规形。程序会报告长度为 1 的重复环。

一般而言，“该确定策略检测到环”不证明别的策略也不能终止；“燃料用尽”更只是尚未得到结果。K x (WWW) 会先丢弃第二个参数并得到 x。

通用等式定理仍能实例化为发散项：证书先用新鲜原子建成，再结构性替换为任意项，不要求这些项归约终止。

In [9]:
omega = parse("W W W")
show_trace("W W W")
show_trace("W W W", fuel=4, detect_cycles=False)
show_trace("K x (W W W)")
omega_proof = associativity_proof(omega, omega, omega)
assert check(omega_proof) == (dot(dot(omega, omega), omega), dot(omega, dot(omega, omega)))
print("Associativity with three divergent arguments: finite certificate checked")
try:
    check(Proof("eta", (app(B, I), I)))
except InvalidProof:
    print("Invented eta rule rejected as expected")
else:
    raise AssertionError("kernel incorrectly accepted eta")


start: W W W  [expanded: W W W]
  01. W@root: W W W
status=cycle; contractions=1
cycle_start=0; cycle_length=1
start: W W W  [expanded: W W W]
  01. W@root: W W W
  02. W@root: W W W
  03. W@root: W W W
  04. W@root: W W W
status=fuel_exhausted; contractions=4
start: K x (W W W)  [expanded: K x (W W W)]
  01. K@root: x
status=normal; contractions=1
Associativity with three divergent arguments: finite certificate checked
Invented eta rule rejected as expected


## 9. 可重复测试与边界

测试涵盖解析、括号、四规则、上下文路径、过量应用、I 的定义、有限序列、16 条已饱和公理、p.521 BW 差异、错误证明拒绝、发散实参实例化，以及固定种子的随机轨迹。另见随包 review_independent.py 的独立结构匹配审查。

这些结果证明的是程序在所列检查中的行为与具体证书被小型内核接受，不能升级为整部原论的一致性／完备性证明。没有实现第二部分的组合表示、正规／适当组合子分类等总定理。

In [10]:
import io
import unittest
import test_curry
suite = unittest.defaultTestLoader.loadTestsFromModule(test_curry)
result = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(suite)
print(f"Unit tests: {result.testsRun}; failures: {len(result.failures)}; errors: {len(result.errors)}")
assert result.wasSuccessful()
print("PASS")


Unit tests: 34; failures: 0; errors: 0
PASS


## 关联阅读

- [复现说明与运行边界](README.md)
- [第一部分中文深读](../../papers/1930/curry-foundations-part1.zh-CN.md)
- [逐页来源与异文核验记录](../../papers/1930/curry-foundations-part1.sources.zh-CN.md)
